In [1]:
from pathlib import Path
import sys
import litellm

# Add the repo root to sys.path so we can import our modules
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from GenerateKeywordCards2.swow import get_swow_rows
from GenerateKeywordCards2.swow import SWOWAssociations

from GenerateKeywordCards2.embeddings import (
    WordEmbeddings,
    compare_with_swow,
    summarize_swow_comparisons,
)

In [2]:
litellm.cache = litellm.Cache(type="disk")


def get_cache_count() -> int:
    disk_cache_object = litellm.cache.cache.disk_cache
    return len(disk_cache_object)


initial_cache_count = get_cache_count()

# SWOW word association dataset

In [3]:
rows = get_swow_rows()
print(f"{len(rows):,} rows")
for row in rows[:5]:
    row_str = ", ".join(f"{k}={v}" for k, v in row.items())
    print(row_str)

Using cached SWOW data/workspaces/SoCloverAI/GenerateKeywordCards2/cache/SWOW-EN.complete.20180827.csv
1,356,362 rows
=1, id=1500428, participantID=130332, created_at=2018-01-07 04:29:38, age=61, nativeLanguage=United States, gender=Ma, education=5, city=Pepperell, country=United States, section=seed, cue=there, R1Raw=position, R2Raw=place, R3Raw=point, R1=position, R2=place, R3=point
=2, id=1500426, participantID=130332, created_at=2018-01-07 04:29:38, age=61, nativeLanguage=United States, gender=Ma, education=5, city=Pepperell, country=United States, section=seed, cue=true, R1Raw=honest, R2Raw=fact, R3Raw=indisputable, R1=honest, R2=fact, R3=indisputable
=3, id=1500424, participantID=130332, created_at=2018-01-07 04:29:38, age=61, nativeLanguage=United States, gender=Ma, education=5, city=Pepperell, country=United States, section=seed, cue=beat, R1Raw=drum, R2Raw=policeman, R3Raw=beatnik, R1=drum, R2=policeman, R3=beatnik
=4, id=1500438, participantID=130332, created_at=2018-01-07 04

In [4]:
print("Corrections to the raw data:")
maximum_corrections_to_list = 25
corrections_count = 0
for row in rows:
    if row["R1Raw"] != row["R1"]:
        print(f"{row['R1Raw']} -> {row['R1']}")
        corrections_count += 1
    if row["R2Raw"] != row["R2"]:
        print(f"{row['R2Raw']} -> {row['R2']}")
        corrections_count += 1
    if row["R3Raw"] != row["R3"]:
        print(f"{row['R3Raw']} -> {row['R3']}")
        corrections_count += 1
    if corrections_count >= maximum_corrections_to_list:
        print(f"Maximum corrections to list ({maximum_corrections_to_list}) reached.")
        break


Corrections to the raw data:
hamdset -> handset
bounf -> bound
american -> American
shrek -> shriek
stident -> student
january -> January
colourless -> colorless
honour -> honor
satan -> Satan
aswell -> as well
abc -> ABC
marvellous -> marvelous
hard labour -> hard labor
labour -> labor
prime mimister -> prime minister
abc -> ABC
dtuy -> duty
Book -> book
Thesis -> thesis
lord of the rings -> Lord of the Rings
In relation to something -> in relation to something
Nearly -> nearly
Not quite -> not quite
In addition to -> in addition to
Abusive -> abusive
Not light handed -> not light handed
Maximum corrections to list (25) reached.


In [5]:
swow = SWOWAssociations()
print(list(swow.associations.keys())[:20])

Using cached SWOW data/workspaces/SoCloverAI/GenerateKeywordCards2/cache/SWOW-EN.complete.20180827.csv
['there', 'position', 'place', 'point', 'true', 'honest', 'fact', 'indisputable', 'beat', 'drum', 'policeman', 'beatnik', 'like', 'affection', 'simile', 'compare', 'telephone', 'receiver', 'handset', 'wires']


In [6]:
print("cat forward associations:")
print(swow.associations["cat"].forward)
print("cat backward associations:")
print(swow.associations["cat"].backward)
print("dog forward associations:")
print(swow.associations["dog"].forward)
print("dog backward associations:")
print(swow.associations["dog"].backward)

cat forward associations:
{'dog': 65, 'feline': 24, 'meow': 17, 'mouse': 14, 'purr': 13, 'fur': 10, 'pet': 9, 'animal': 8, 'soft': 7, 'kitten': 7, 'furry': 6, 'pussy': 6, 'kitty': 5, 'tiger': 5, 'lion': 4, 'hair': 4, 'nap': 4, 'purring': 3, 'scratch': 3, 'black': 3, 'cute': 3, 'Tabby': 3, 'fluffy': 3, 'claws': 3, 'litter': 3, 'house': 2, 'nip': 2, 'category': 2, 'tail': 2, 'cuddly': 2, 'whiskers': 2, 'bitch': 2, 'warm': 2, 'sneaky': 2, 'allergy': 2, 'love': 2, 'hat': 2, 'sat': 2, 'mat': 2, 'fish': 2, 'aloof': 2, 'catwalk': 1, 'predator': 1, 'mice': 1, 'annoying': 1, 'Facebook': 1, 'nuisance': 1, 'chow': 1, 'face': 1, 'hipster': 1, 'machine': 1, 'lovable': 1, 'type of animal': 1, 'fool around': 1, 'burglar': 1, 'adorable': 1, 'box': 1, 'leopard': 1, 'friendly': 1, 'lady': 1, 'crazy': 1, 'bed': 1, 'whisker': 1, 'tin roof': 1, 'o nine tails': 1, 'slippery': 1, 'bitchy': 1, 'strophe': 1, 'paw': 1, 'erpillar': 1, 'carnivore': 1, 'Cheshire': 1, 'puss': 1, 'cuddle': 1, 'Harriet': 1, 'bogus': 

## Embeddings-based associations (via litellm)

Use `GenerateKeywordCards2.embeddings.WordEmbeddings` to:
1. Compare embedding nearest-neighbors against the SWOW human association norms above, to see how well embeddings approximate human associations.
2. Score words with graph-theory-like centrality/diversity metrics to find candidates whose associations span a large, diverse set of other words -- useful for So Clover! keyword expansion.

### Compare embedding neighbors with SWOW human associations

In [7]:
# Build a vocabulary: a handful of cue words plus their SWOW forward responses, plus a
# broader sample of other SWOW words so nearest-neighbor search isn't trivially
# restricted to only the "correct" answers.
sample_cues = ["cat", "dog", "beat", "true", "telephone"]

vocabulary = set(sample_cues)
for cue in sample_cues:
    vocabulary.update(list(swow.associations[cue].forward.keys())[:10])

other_words = list(swow.associations.keys())[:300]
vocabulary.update(other_words)

embedding_models = {
    "openai/text-embedding-3-small",
    "openai/text-embedding-3-large",
    "gemini/gemini-embedding-2",
}
word_embeddings_by_model = {}
for embedding_model in embedding_models:
    word_embeddings, usage = await WordEmbeddings.from_words_async(embedding_model, list(vocabulary))
    word_embeddings_by_model[embedding_model] = word_embeddings
    print(f"* {embedding_model} embedded {len(word_embeddings)} words ({usage})")

* gemini/gemini-embedding-2 embedded 340 words (340/340 cache hits, $0.0000000 uncached cost, $0.0000910 total cost, 455 prompt tokens)
* openai/text-embedding-3-small embedded 340 words (340/340 cache hits, $0.0000000 uncached cost, $0.0000091 total cost, 455 prompt tokens)
* openai/text-embedding-3-large embedded 340 words (340/340 cache hits, $0.0000000 uncached cost, $0.0000592 total cost, 455 prompt tokens)


In [8]:
comparisons = [
    compare_with_swow(word_embeddings_by_model["openai/text-embedding-3-small"], swow, cue, count=10)
    for cue in sample_cues
]
for comparison in comparisons:
    print(f"{comparison.word}: embedding neighbors = {comparison.embedding_neighbors}")
    print(f"{' ' * len(comparison.word)}  swow responses    = {comparison.swow_responses}")
    print(
        f"{' ' * len(comparison.word)}  overlap = {comparison.overlap} (jaccard={comparison.jaccard:.2f})"
    )

print(summarize_swow_comparisons(comparisons))

cat: embedding neighbors = ['feline', 'dog', 'animal', 'pet', 'kitten', 'meow', 'purr', 'car', 'child', 'mouse']
     swow responses    = ['dog', 'feline', 'meow', 'mouse', 'purr', 'fur', 'pet', 'animal', 'soft', 'kitten']
     overlap = ['feline', 'dog', 'animal', 'pet', 'kitten', 'meow', 'purr', 'mouse'] (jaccard=0.67)
dog: embedding neighbors = ['animal', 'pet', 'canine', 'cat', 'puppy', 'child', 'mouse', 'dad', 'hand', 'bow']
     swow responses    = ['cat', 'pet', 'friend', 'canine', 'puppy', 'animal', 'bark', 'bone', 'fur', 'woof']
     overlap = ['animal', 'pet', 'canine', 'cat', 'puppy'] (jaccard=0.33)
beat: embedding neighbors = ['bang', 'hit', 'beatnik', 'touch', 'drive', 'music', 'rhythm', 'walk', 'moment', 'shoot']
      swow responses    = ['drum', 'music', 'hit', 'rhythm', 'win', 'up', 'eggs', 'it', 'heart', 'nick']
      overlap = ['hit', 'music', 'rhythm'] (jaccard=0.18)
true: embedding neighbors = ['false', 'correct', 'truth', 'real', 'fact', 'complete', 'full', 'lies'

### Candidate selection: words whose associations span diverse concepts

In [9]:
# A high bridging score means a word's nearest neighbors are both numerous (high
# centrality) and mutually diverse (they span different concepts) -- a candidate
# keyword that can plausibly connect to many different other keywords.
bridging_scores = word_embeddings_by_model[
    "openai/text-embedding-3-small"
].bridging_scores(k=8)
top_bridging_words = sorted(bridging_scores.items(), key=lambda item: item[1], reverse=True)[:15]
for word, score in top_bridging_words:
    print(f"{word:<15} {score:.3f}")

car             0.737
head            0.717
beat            0.709
front           0.708
hard            0.683
reduce          0.676
first           0.668
dark            0.662
capture         0.661
complete        0.659
delay           0.656
information     0.655
eat             0.649
activity        0.649
food            0.647


In [10]:
final_cache_count = get_cache_count()
print(
    f"Cache count {final_cache_count}, new entries this run: {final_cache_count - initial_cache_count}"
)

Cache count 3170, new entries this run: 0
